# Content Moderation with governed-rank

Toxic content is engaging — outrage drives clicks. Every content platform faces the
same dilemma: the engagement model rewards toxicity.

This notebook shows how `govern()` demotes toxic content **without destroying
engagement quality**. The key: orthogonalization removes the engagement-toxicity
correlation before steering.

**What you'll see:**
- The engagement-toxicity trap (correlation by design)
- Why naive penalties over-correct and break the ranking
- How MOSAIC steers efficiently — targeting only the uncertain zone
- The budget knob: one parameter controls the tradeoff

In [ ]:
import numpy as np
from scipy.stats import kendalltau
import matplotlib.pyplot as plt
from mosaic import govern

# ---------------------------------------------------------------------------
# Generate a realistic content feed
# ---------------------------------------------------------------------------
# Toxic content is engaging: a latent "outrage" factor drives both signals.
# This mirrors reality: inflammatory content drives clicks, shares, comments.

np.random.seed(42)
n = 200

# Latent outrage factor (Beta-distributed: most posts are mild)
outrage = np.random.beta(2, 5, n)

# Engagement: base quality + outrage boost
quality = np.random.uniform(0.3, 1.0, n)
engagement = 0.6 * quality + 0.4 * outrage + np.random.normal(0, 0.05, n)
engagement = np.clip(engagement, 0.01, 1.0)

# Toxicity: 0 = safe, 1 = toxic
toxicity = 0.7 * outrage + 0.3 * np.random.uniform(0, 0.4, n)
toxicity = np.clip(toxicity, 0, 1)

# Steering signal: safety score (higher = safer = promote)
safety = 1.0 - toxicity

r = np.corrcoef(engagement, toxicity)[0, 1]
print(f"Posts generated:       {n}")
print(f"Corr(engage, toxic):   {r:.3f}")
print(f"Mean toxicity:         {toxicity.mean():.3f}")
print(f"Toxic posts (>0.3):    {(toxicity > 0.3).sum()} / {n} ({100*(toxicity > 0.3).mean():.0f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
safe_mask = toxicity <= 0.3
ax.scatter(engagement[safe_mask], toxicity[safe_mask],
           c='#2ca02c', alpha=0.5, s=25, label='Safe (\u22640.3)')
ax.scatter(engagement[~safe_mask], toxicity[~safe_mask],
           c='#d62728', alpha=0.6, s=25, label='Toxic (>0.3)')
ax.axhline(y=0.3, color='gray', linestyle='--', alpha=0.4)
ax.set_xlabel('Engagement Score')
ax.set_ylabel('Toxicity Score')
ax.set_title(f'The Engagement-Toxicity Trap (r = {r:.2f})')
ax.legend()
plt.tight_layout()
plt.show()

## 1. Base Ranking: Engagement Only

When we rank purely by engagement, toxic posts float to the top because
outrage drives engagement. This is the problem every content platform faces.

We take the top 50 posts for ranking (simulating one page of a content feed).
`govern()`'s internal `max_rank=50` covers all edges, so `budget=1.0` correctly
returns the base ranking unchanged.

In [ ]:
# Select the top 50 posts by engagement for our content feed.
# With 50 items, govern()'s max_rank=50 covers all edges.
top_idx = np.argsort(-engagement)[:50]

# Steering weight: scales safety signal relative to engagement range.
steer_weight = 0.5

base_scores  = {int(i): float(engagement[i]) for i in top_idx}
steer_scores = {int(i): steer_weight * float(safety[i]) for i in top_idx}
tox_lookup   = {int(i): float(toxicity[i]) for i in top_idx}

base_order = sorted(base_scores, key=lambda x: -base_scores[x])

print("BASE RANKING (top 10 by engagement)")
print("=" * 60)
toxic_base_10 = 0
for rank, idx in enumerate(base_order[:10]):
    is_tox = tox_lookup[idx] > 0.3
    toxic_base_10 += is_tox
    tag = " << TOXIC" if is_tox else ""
    print(f"  {rank+1:2d}. Post {idx:3d}   eng={base_scores[idx]:.3f}   tox={tox_lookup[idx]:.3f}{tag}")

print(f"\nToxic in top 10: {toxic_base_10}/10 ({100*toxic_base_10/10:.0f}%)")

## 2. Naive Fix: Subtract Toxicity

The obvious approach: penalize toxic content by subtracting toxicity from engagement.

```
final = engagement - 0.5 * toxicity
```

This seems reasonable — toxic posts get lower scores. Watch what happens to ranking quality.

In [ ]:
# Naive: subtract toxicity penalty from engagement.
# Because toxicity correlates with engagement, this reshuffles the ENTIRE ranking,
# not just the toxic posts.
penalty_weight = 0.5
naive_scores = {i: base_scores[i] - penalty_weight * tox_lookup[i] for i in base_scores}
naive_order = sorted(naive_scores, key=lambda x: -naive_scores[x])

# Kendall tau: rank correlation with base ordering.
#   tau = 1.0 -> identical ordering
#   tau = 0   -> random
#   tau = -1  -> reversed
# Quality retained = (1 + tau) / 2
tau_naive, _ = kendalltau(
    [base_order.index(i) for i in base_order],
    [naive_order.index(i) for i in base_order]
)

naive_toxic_10 = sum(1 for i in naive_order[:10] if tox_lookup[i] > 0.3)

print("NAIVE RANKING (top 10)")
print("=" * 60)
for rank, idx in enumerate(naive_order[:10]):
    tag = " << TOXIC" if tox_lookup[idx] > 0.3 else ""
    print(f"  {rank+1:2d}. Post {idx:3d}   eng={base_scores[idx]:.3f}   "
          f"tox={tox_lookup[idx]:.3f}   naive={naive_scores[idx]:.3f}{tag}")

print(f"\nToxic in top 10: {naive_toxic_10}/10")
print(f"Kendall tau vs base: {tau_naive:.3f}")
print(f"Quality retained: {100*(1+tau_naive)/2:.1f}%")

## 3. MOSAIC: Orthogonalize, Then Steer

`govern()` removes the engagement-toxicity correlation **before** steering.
The remaining safety signal can only move posts where the engagement model
is uncertain — it cannot demote posts that are clearly both engaging AND safe.

**Key diagnostics:**
- **Projection coefficient**: how much correlation was removed (negative means
  safety was anti-correlated with engagement, as expected for toxic=engaging)
- **Protected edges**: ordering constraints the budget locked
- **Active constraints**: how many of those constraints actually bound

In [ ]:
result = govern(base_scores, steer_scores, budget=0.30)

mosaic_order = result.ranked_items
mosaic_toxic_10 = sum(1 for i in mosaic_order[:10] if tox_lookup[i] > 0.3)

tau_mosaic, _ = kendalltau(
    [base_order.index(i) for i in base_order],
    [mosaic_order.index(i) for i in base_order]
)

print("MOSAIC RANKING (top 10)")
print("=" * 60)
for rank, idx in enumerate(mosaic_order[:10]):
    tag = " << TOXIC" if tox_lookup[idx] > 0.3 else ""
    print(f"  {rank+1:2d}. Post {idx:3d}   eng={base_scores[idx]:.3f}   tox={tox_lookup[idx]:.3f}{tag}")

print(f"\nToxic in top 10: {mosaic_toxic_10}/10")
print(f"Kendall tau vs base: {tau_mosaic:.3f}")
print(f"Quality retained: {100*(1+tau_mosaic)/2:.1f}%")
print(f"Projection coefficient: {result.projection_coeff:.4f}")
print(f"Protected edges: {result.n_protected_edges}")
print(f"Active constraints: {result.n_active_constraints}")

## 4. Head-to-Head Comparison

How much toxicity does each method remove, and at what cost to engagement quality?

| Metric | What it measures |
|--------|------------------|
| Toxic/K | Number of toxic posts (score > 0.3) in top K |
| MeanTox | Average toxicity score of top-10 posts |
| Kendall tau | Rank correlation with base (1.0 = identical ordering) |
| Quality | (1 + tau) / 2 — fraction of pairwise orderings preserved |

In [ ]:
print(f"{'':>8} {'Toxic/5':>8} {'Toxic/10':>9} {'Toxic/25':>9} {'MeanTox':>8} {'Tau':>7} {'Quality':>8}")
print("-" * 58)

for name, order, tau in [
    ("Base",   base_order,   1.0),
    ("Naive",  naive_order,  tau_naive),
    ("MOSAIC", mosaic_order, tau_mosaic),
]:
    t5  = sum(1 for i in order[:5]  if tox_lookup[i] > 0.3)
    t10 = sum(1 for i in order[:10] if tox_lookup[i] > 0.3)
    t25 = sum(1 for i in order[:25] if tox_lookup[i] > 0.3)
    mt  = np.mean([tox_lookup[i] for i in order[:10]])
    q   = (1 + tau) / 2
    print(f"{name:>8} {t5:>8} {t10:>9} {t25:>9} {mt:>8.3f} {tau:>7.3f} {100*q:>7.1f}%")

# Mean toxicity comparison
base_mt  = np.mean([tox_lookup[i] for i in base_order[:10]])
naive_mt = np.mean([tox_lookup[i] for i in naive_order[:10]])
mosaic_mt = np.mean([tox_lookup[i] for i in mosaic_order[:10]])

print(f"\nMean toxicity in top 10:")
print(f"  Base:   {base_mt:.3f}")
print(f"  Naive:  {naive_mt:.3f}  ({100*(naive_mt - base_mt)/base_mt:+.1f}%)")
print(f"  MOSAIC: {mosaic_mt:.3f}  ({100*(mosaic_mt - base_mt)/base_mt:+.1f}%)")

## 5. Budget Sweep

The budget parameter controls the accuracy-safety tradeoff with a single knob:

| Budget | Meaning |
|--------|--------|
| 0.00 | No edges protected — maximum safety steering |
| 0.30 | Protect 30% most confident gaps (recommended default) |
| 1.00 | All edges protected — output identical to base ranking |

In [ ]:
# budget = 0.00: no edges protected, maximum steering
# budget = 0.30: protect 30% most confident gaps (default)
# budget = 1.00: all edges protected → output = base ranking (tau = 1.0)

print(f"{'Budget':>8} {'Toxic/10':>9} {'MeanTox':>9} {'Tau':>8} {'Quality':>9}")
print("-" * 48)

for b in [0.00, 0.10, 0.20, 0.30, 0.50, 0.70, 1.00]:
    r = govern(base_scores, steer_scores, budget=b)
    order = r.ranked_items
    t10 = sum(1 for i in order[:10] if tox_lookup[i] > 0.3)
    mt = np.mean([tox_lookup[i] for i in order[:10]])
    tau, _ = kendalltau(
        [base_order.index(i) for i in base_order],
        [order.index(i) for i in base_order]
    )
    q = (1 + tau) / 2
    print(f"{b:>8.2f} {t10:>9} {mt:>9.3f} {tau:>8.3f} {100*q:>8.1f}%")

print("\nAt budget=1.00, tau=1.000 — the ranking is identical to base (all edges protected).")
print("As budget decreases, more steering takes effect and toxicity in the top-10 drops.")

## Summary

| | Base | Naive | MOSAIC |
|---|---|---|---|
| **Approach** | Rank by engagement | Subtract toxicity penalty | Orthogonalize + constrained projection |
| **Toxicity** | High (outrage wins) | Reduced (but unpredictable) | Reduced (targeted at uncertain zone) |
| **Quality** | Perfect (baseline) | Degraded (reshuffles everything) | Preserved (only moves uncertain items) |
| **Control** | None | Weight parameter (fragile) | Budget knob (smooth, monotonic) |

**Key insight**: Naive toxicity penalties subtract a signal that's correlated with
engagement, causing unpredictable reshuffling. MOSAIC orthogonalizes first —
the remaining safety signal can only move posts where the engagement model doesn't
have a strong opinion. This is why it achieves the same (or better) toxicity
reduction with less quality loss.

```python
from mosaic import govern
result = govern(engagement_scores, safety_scores, budget=0.30)
```